In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

class DeepMLP(nn.Module):
    def __init__(self, activation="relu"):
        super().__init__()
        act = nn.ReLU if activation == "relu" else nn.Sigmoid
        self.net = nn.Sequential(
            nn.Flatten(), nn.Linear(784, 256), act(),
            nn.Linear(256,256), act(), nn.Linear(256,256), act(),
            nn.Linear(256,256), act(), nn.Linear(256,10)
        )
    def forward(self, x): return self.net(x)
    
def run_experiment(title, activation="relu", lr=0.001, max_batches=100):
    print("\n====", title, "====")
    model = DeepMLP(activation).to(device)
    criterion = nn.CrossEntropyLoss()
    optim = torch.optim.SGD(model.parameters(), lr=lr)
    for batch_idx, (X, y) in enumerate(train_loader):
        if batch_idx >= max_batches: break
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = criterion(pred, y)
        optim.zero_grad()
        loss.backward()
        grad_norms = [p.grad.norm().item() for n, p in model.named_parameters() if p.grad is not None and "weight" in n]
        optim.step()
        if batch_idx % 20 == 0:
            print(f"batch={batch_idx}, loss={loss.item():.4f}, grad_norms={[round(g, 6) for g in grad_norms]}")
            
run_experiment("正常训练: ReLU + lr = 0.01", "relu", 0.01)
run_experiment("可能梯度爆炸: ReLU + lr = 5.0", "relu", 5.0)
run_experiment("可能梯度消失: Sigmoid + lr = 0.01", "sigmoid", 0.01)


==== 正常训练: ReLU + lr = 0.01 ====
batch=0, loss=2.2994, grad_norms=[0.024482, 0.012871, 0.013546, 0.018223, 0.03061]
batch=20, loss=2.2984, grad_norms=[0.024969, 0.013391, 0.014624, 0.017285, 0.026207]
batch=40, loss=2.3042, grad_norms=[0.025442, 0.013524, 0.013617, 0.019122, 0.032272]
batch=60, loss=2.3023, grad_norms=[0.024903, 0.013341, 0.013648, 0.019736, 0.034306]
batch=80, loss=2.3031, grad_norms=[0.024632, 0.012893, 0.012555, 0.015807, 0.026873]

==== 可能梯度爆炸: ReLU + lr = 5.0 ====
batch=0, loss=2.3034, grad_norms=[0.02902, 0.015882, 0.017166, 0.017576, 0.029397]
batch=20, loss=nan, grad_norms=[nan, nan, nan, nan, nan]
batch=40, loss=nan, grad_norms=[nan, nan, nan, nan, nan]
batch=60, loss=nan, grad_norms=[nan, nan, nan, nan, nan]
batch=80, loss=nan, grad_norms=[nan, nan, nan, nan, nan]

==== 可能梯度消失: Sigmoid + lr = 0.01 ====
batch=0, loss=2.3488, grad_norms=[0.000532, 0.002536, 0.019804, 0.138182, 0.922931]
batch=20, loss=2.3021, grad_norms=[0.000503, 0.001924, 0.013042, 0.095987,